# Search 25,000 Museum Images With One Sentence

Type a natural-language query and retrieve matching Smithsonian images with precomputed CLIP image embeddings. The image embeddings are optional convenience artifacts; Smithsonian metadata and images remain the dataset source-of-truth.

## Setup

The dataset ships the 24,972 image embeddings, so this notebook only encodes the query text. On a fresh Kaggle session the Hugging Face CLIP text/model weights must be available; enable Internet for the first model download if they are not already cached.

In [ ]:
from pathlib import Path
import io
import json
import zipfile
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from PIL import Image

roots = list(Path('/kaggle/input').glob('*/metadata.parquet'))
DATA_ROOT = roots[0].parent if roots else Path('../data/release/museum-images').resolve()
metadata = pd.read_parquet(DATA_ROOT / 'metadata.parquet').set_index('image_id', drop=False)
optional_dir = DATA_ROOT / 'optional'
if optional_dir.is_dir():
    embedding_table = pq.read_table(optional_dir / 'clip_embeddings.parquet')
    embedding_manifest = json.loads((optional_dir / 'clip_embeddings_manifest.json').read_text())
else:
    with zipfile.ZipFile(DATA_ROOT / 'optional.zip') as archive:
        embedding_table = pq.read_table(io.BytesIO(archive.read('clip_embeddings.parquet')))
        embedding_manifest = json.loads(archive.read('clip_embeddings_manifest.json').decode('utf-8'))
embedding_ids = np.asarray(embedding_table['image_id'].to_numpy(), dtype=np.int64)
image_embeddings = np.asarray(embedding_table['embedding'].to_pylist(), dtype=np.float32)
MODEL_ID = embedding_manifest['model_id']
MODEL_REVISION = embedding_manifest.get('model_revision')
IMAGE_DIR = DATA_ROOT / 'images'
IMAGE_ZIP = DATA_ROOT / 'images.zip'
IMAGE_ARCHIVE = zipfile.ZipFile(IMAGE_ZIP) if (not IMAGE_DIR.is_dir() and IMAGE_ZIP.is_file()) else None

def load_release_image(file_name):
    if IMAGE_DIR.is_dir():
        return Image.open(IMAGE_DIR / file_name).convert('RGB')
    if IMAGE_ARCHIVE is None:
        raise FileNotFoundError('Neither images/ nor images.zip was found')
    return Image.open(io.BytesIO(IMAGE_ARCHIVE.read(file_name))).convert('RGB')
print('Rows:', len(metadata), 'Embedding matrix:', image_embeddings.shape)
print('Model:', MODEL_ID, 'revision:', embedding_manifest.get('model_revision'))

In [ ]:
assert len(metadata) == len(embedding_ids) == len(image_embeddings)
assert set(metadata.index.astype(int)) == set(embedding_ids.tolist())
norms = np.linalg.norm(image_embeddings, axis=1)
assert np.max(np.abs(norms - 1.0)) < 0.01
print('Embedding integrity checks passed.')

## Load the CLIP text encoder

In [ ]:
import torch
from transformers import AutoTokenizer, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = CLIPModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).eval().to(device)
print('Device:', device)

## Search

Try `a spacecraft`, `a dinosaur fossil`, `a Japanese ceramic bowl`, or your own sentence.

In [ ]:
QUERY = 'a spacecraft'
TOP_K = 12

tokens = tokenizer([QUERY], padding=True, truncation=True, return_tensors='pt').to(device)
with torch.inference_mode():
    query_embedding = model.get_text_features(**tokens).float()
    query_embedding = query_embedding / query_embedding.norm(dim=1, keepdim=True)
query_embedding = query_embedding.cpu().numpy()[0]
scores = image_embeddings @ query_embedding
top_positions = np.argsort(scores)[-TOP_K:][::-1]
top_ids = embedding_ids[top_positions]
results = metadata.loc[top_ids].copy()
results['clip_score'] = scores[top_positions]
results[['image_id', 'title', 'category', 'institution', 'clip_score']]

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, (_, row) in zip(axes.flat, results.iterrows()):
    image = load_release_image(row['file_name'])
    ax.imshow(image)
    label = f"{row['clip_score']:.3f} — {str(row['title'])[:60]}"
    ax.set_title(label, fontsize=8)
    ax.axis('off')
plt.suptitle(QUERY, fontsize=15)
plt.tight_layout()
plt.show()

## What the score means

Both image and text vectors are L2-normalized, so the dot product is cosine similarity in the shared CLIP embedding space. This is a zero-shot retrieval baseline, not a claim that the model fully understands museum context or cultural meaning.

## Leakage-safe evaluation

For quantitative experiments, use the provided `split` column. All views of a Smithsonian `object_id` stay in one split, so front/back/detail views cannot cross train, validation, and test.